# CMSC 173 &middot; Module 8 &middot; Example 2
## Penguins: KNN, and why scaling matters

333 penguins of three species, three body measurements. KNN classifies by distance, and a distance that adds grams to millimetres is mostly grams.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides; change something and see what moves.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

penguins = sns.load_dataset("penguins").dropna()          # 333 penguins with every measurement
X = penguins[["bill_length_mm", "flipper_length_mm", "body_mass_g"]]
y = penguins["species"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=173, stratify=y)

raw = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_tr, y_tr)
print(round(raw.score(X_te, y_te), 2), round(scaled.score(X_te, y_te), 2))   # 0.72 0.96

**Why.** One standard deviation of body mass is 805 g; of bill length, 5.5 mm. Unscaled, a distance is almost all grams, and Adelie and Chinstrap weigh the same.

In [ ]:
print(X.std().round(1).to_dict())
print(penguins.groupby("species")["body_mass_g"].mean().round())   # Adelie 3706, Chinstrap 3733, Gentoo 5092
labels = ["Adelie", "Chinstrap", "Gentoo"]
print(confusion_matrix(y_te, raw.predict(X_te), labels=labels))     # Chinstrap: only 8 of 20 found
print(confusion_matrix(y_te, scaled.predict(X_te), labels=labels))  # Chinstrap: 19 of 20

In [ ]:
sns.scatterplot(data=penguins, x="body_mass_g", y="bill_length_mm", hue="species", style="species")
plt.show()

**Choosing k.** Five-fold cross-validation on the training rows, as in module 7: almost flat from 1 to 25.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=173)
for k in (1, 3, 5, 9, 15, 25, 51):
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    print(k, round(cross_val_score(model, X_tr, y_tr, cv=cv).mean(), 3))

**All three methods on the penguins.** Five-fold cross-validation on all 333 (Part 10 of the lecture).

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

models = {"naive Bayes": GaussianNB(),
          "KNN, k=5, scaled": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
          "tree, depth 3": DecisionTreeClassifier(max_depth=3, random_state=173),
          "KNN, k=5, unscaled": KNeighborsClassifier(n_neighbors=5)}
for name, m in models.items():
    s = cross_val_score(m, X, y, cv=cv)
    print(f"{name:20s} {s.mean():.3f} ± {s.std():.3f}   test {m.fit(X_tr, y_tr).score(X_te, y_te):.2f}")

**Try it.** Drop `body_mass_g` from `X` and refit the *unscaled* model. Why does accuracy go up?